# Background to signal flavour tags
## Calculate background to signal ratios for flavour double tags

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [1]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [2]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/math.hpp>
#include<uncertainties/stat.hpp>

### Load utility functions

In [3]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Number of bins

In [4]:
const int NumberBins = 4;

### Get reconstructed background bin yields

In [5]:
std::map<int, double> GetRecBackgroundBinYields(const std::string &TagMode,
                                                const std::string &BackgroundMode) {
    std::string Filename = "${BES3_ANALYSIS_PATH}/Selection/PeakingBackgrounds/DoubleTag/";
    Filename += TagMode + "/KKpipi_vs_" + BackgroundMode + "_to_KKpipi_vs_";
    Filename += TagMode + "_DoubleTag_SignalMC_Binned.root";
    TChain Chain((TagMode + "DoubleTag").c_str());
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, true, NumberBins);
}

### Get generated bin yields

In [6]:
std::map<int, double> GetGenBinYields(const std::string &TagMode) {
    std::string Filename = "${BES3_ANALYSIS_PATH}/TruthTuples/BinnedTruthTuples/";
    Filename += TagMode + "/KKpipi_vs_" + TagMode + "_TruthTuple_Binned.root";
    TChain Chain("TruthTuple");
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, false, NumberBins);
}

### List of tag modes and their backgrounds

In [7]:
std::map<std::string, std::vector<std::string>> TagModes{
    {"Kpipipi", {"KSKpi"}},
    {"KeNu", {"Kpi0eNu", "KmuNu", "Kpipi0"}}
};

### Start calculating the background to signal bin efficiencies, times the ratio of branching fractions

In [8]:
std::string BackgroundToSignalRatios;
for(const auto &Tag : TagModes) {
    for(std::size_t i = 0; i < Tag.second.size(); i++) {
        const auto SignalBF = GetBranchingFraction(Tag.first);
        uncertainties::udouble SignalBF_unc(SignalBF.first, SignalBF.second);
        const auto BackgroundBF = GetBranchingFraction(Tag.second[i]);
        uncertainties::udouble BackgroundBF_unc(BackgroundBF.first, BackgroundBF.second);
        const auto BFRatio = BackgroundBF_unc/SignalBF_unc;
        const auto SignalRecYields = GetRecSignalBinYields(Tag.first, NumberBins);
        const auto SignalGenYields = GetGenBinYields(Tag.first);
        const auto BackgroundRecYields = GetRecBackgroundBinYields(Tag.first, Tag.second[i]);
        const auto BackgroundGenYields = GetGenBinYields(Tag.second[i]);
        std::vector<uncertainties::udouble> BkgToSigRatio;
        for(int Bin = -NumberBins; Bin <= NumberBins; Bin++) {
            if(Bin == 0) {
                continue;
            }
            std::string Label = Tag.first + "_PeakingBackground" + std::to_string(i + 1);
            Label += "_DoubleTag_Flavour_KKpipi_vs_" + Tag.first + "_SignalBin";
            Label += (Bin > 0 ? "P" : "M") + std::to_string(TMath::Abs(Bin));
            Label += "_TagBin0_BackgroundToSignalRatio";
            const double SigEff = SignalRecYields.at(Bin)/SignalGenYields.at(Bin);
            const double SigEff_err = TMath::Sqrt(SigEff*(1.0 - SigEff)/SignalGenYields.at(Bin));
            const uncertainties::udouble SigEff_unc(SigEff, SigEff_err);
            const double BkgEff = BackgroundRecYields.at(Bin)/BackgroundGenYields.at(Bin);
            const double BkgEff_err = TMath::Sqrt(BkgEff*(1.0 - BkgEff)/BackgroundGenYields.at(Bin));
            const uncertainties::udouble BkgEff_unc(BkgEff, BkgEff_err);
            const auto EffRatio = BkgEff_unc/SigEff_unc;
            BkgToSigRatio.push_back(EffRatio*BFRatio);
            BackgroundToSignalRatios += Label + " ";
            BackgroundToSignalRatios += std::to_string(uncertainties::nom(BkgToSigRatio.back())) + "\n";
            BackgroundToSignalRatios += Label + "_err ";
            BackgroundToSignalRatios += std::to_string(uncertainties::sdev(BkgToSigRatio.back())) + "\n";
        }
        BackgroundToSignalRatios += "\n";
        std::string Filename = "PeakingBackground_DT_" + Tag.second[i] + "_to_" + Tag.first + ".root";
        std::vector<double> FlatCovMatrix =
            uncertainties::cov_matrix<std::vector<double>>(BkgToSigRatio);
        SaveCovMatrix(FlatCovMatrix, Filename);
    }
}
std::cout << BackgroundToSignalRatios;

KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinM4_TagBin0_BackgroundToSignalRatio 0.077893
KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinM4_TagBin0_BackgroundToSignalRatio_err 0.002692
KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinM3_TagBin0_BackgroundToSignalRatio 0.078732
KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinM3_TagBin0_BackgroundToSignalRatio_err 0.001571
KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinM2_TagBin0_BackgroundToSignalRatio 0.076370
KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinM2_TagBin0_BackgroundToSignalRatio_err 0.001742
KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinM1_TagBin0_BackgroundToSignalRatio 0.077003
KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinM1_TagBin0_BackgroundToSignalRatio_err 0.002467
KeNu_PeakingBackground1_DoubleTag_Flavour_KKpipi_vs_KeNu_SignalBinP1_TagBin0_BackgroundToSignalRatio 0.0

### Save parameters to a file

In [9]:
std::ofstream File("BackgroundToSignalRatios_Flavour.txt");
File << BackgroundToSignalRatios;
File.close();